In [165]:
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt

In [166]:
# Merge files - set path to scraped files
file_paths = glob.glob('./data/basel_*.csv')

# Load all files and concatenate them into a single DataFrame, handle special characters if any
dataframes = [pd.read_csv(file, encoding='ISO-8859-1') for file in file_paths]
merged_data = pd.concat(dataframes, ignore_index=True)

# Separate Arrivals and Departures
arrivals_data = merged_data[merged_data['Type'] == 'Arrivals']
departures_data = merged_data[merged_data['Type'] == 'Departures']

# Concatenate Arrivals first and then Departures
final_data = pd.concat([arrivals_data, departures_data], ignore_index=True)

print(final_data)

            Type   Time Expected       Origin/Destination  \
0       Arrivals   5:15     5:29  Paris Charles De Gaulle   
1       Arrivals   5:15     5:11                    Liege   
2       Arrivals   5:15     5:14                 Brussels   
3       Arrivals   5:20     5:18             Cologne Bonn   
4       Arrivals   5:40     5:54             Cologne Bonn   
...          ...    ...      ...                      ...   
2748  Departures  20:25      NaN                Barcelona   
2749  Departures  20:40      NaN                  Leipzig   
2750  Departures  21:00      NaN                   Lisbon   
2751  Departures  21:10      NaN           London Gatwick   
2752  Departures  21:10      NaN                     Doha   

                           Airline Flight Number           Status   Note  \
0                            FedEx        FX6257     Landed 05:27  Cargo   
1                            FedEx        FX4862     Landed 05:07  Cargo   
2     European Air Transport / DHL     

In [167]:
# Copy dataframe
df = final_data.copy()

# Drop the irrelevant columns 
df.drop(columns=['Note'], inplace=True)
df.drop(columns=['Flight Number'], inplace=True)

# Add column LocalAirport
df['LocalAirport'] = 'Basel'

In [168]:

# Remove rows where Status is 'cancelled', 'boarding closed', or is empty
df = df[df['Status'].str.strip() != "Cancelled"]
df = df[~df['Status'].str.lower().isin(['cancelled', 'boarding closed']) &
        df['Status'].notna() & (df['Status'] != '')]



In [169]:
# Check for missing data in all columns
missing_data_summary = df.isna().sum()
print("Missing Data Summary:")
print(missing_data_summary)

Missing Data Summary:
Type                     0
Time                     0
Expected              1161
Origin/Destination       0
Airline                  6
Status                   0
Date Scraped             0
LocalAirport             0
dtype: int64


In [170]:
# Extract the time part (HH:MM) from 'Status' and store it in a new column 'Actual' 
df['Actual'] = df['Status'].str.extract(r'(\d{2}:\d{2})')
df

,Type,Time,Expected,Origin/Destination,Airline,Status,Date Scraped,LocalAirport,Actual
0,Arrivals,5:15,5:29,Paris Charles De Gaulle,FedEx,Landed 05:27,07-11-24 23:55,Basel,05:27
1,Arrivals,5:15,5:11,Liege,FedEx,Landed 05:07,07-11-24 23:55,Basel,05:07
2,Arrivals,5:15,5:14,Brussels,European Air Transport / DHL,Landed 05:11,07-11-24 23:55,Basel,05:11
3,Arrivals,5:20,5:18,Cologne Bonn,Sprintair,Landed 05:16,07-11-24 23:55,Basel,05:16
4,Arrivals,5:40,5:54,Cologne Bonn,Sprintair,Landed 05:51,07-11-24 23:55,Basel,05:51
...,...,...,...,...,...,...,...,...,...
2748,Departures,20:25,NaN,Barcelona,easyJet,Departed 20:21,09-11-24 23:55,Basel,20:21
2749,Departures,20:40,NaN,Leipzig,European Air Transport / DHL,Departed 20:57,09-11-24 23:55,Basel,20:57
2750,Departures,21:00,NaN,Lisbon,easyJet,Departed 21:00,09-11-24 23:55,Basel,21:00
2751,Departures,21:10,NaN,London Gatwick,easyJet,Departed 21:13,09-11-24 23:55,Basel,21:13


In [171]:
# Function to parse dates with mixed formats and adjust for two-digit years
def transform_date(date_string):
    # Split on the first space to get the date part
    date_part = date_string.split()[0]
    if len(date_part) == 10:  # Format YYYY-MM-DD
        return date_part
    elif len(date_part) == 8:  # Format DD-MM-YY
        # Transform YY to YYYY assuming last two digits represent the year
        day, month, year = date_part.split("-")
        year = "20" + year  # Convert YY to YYYY
        return f"{year}-{month}-{day}"
    else:
        return None  # Handle unexpected format if necessary

# Apply the transformation to create the new column
df["ActualDate"] = pd.to_datetime(df["Date Scraped"].apply(transform_date) + ' ' + df['Actual'])
df["ExpectedDate"] = pd.to_datetime(df["Date Scraped"].apply(transform_date) + ' ' + df['Expected'])

df

,Type,Time,Expected,Origin/Destination,Airline,Status,Date Scraped,LocalAirport,Actual,ActualDate,ExpectedDate
0,Arrivals,5:15,5:29,Paris Charles De Gaulle,FedEx,Landed 05:27,07-11-24 23:55,Basel,05:27,2024-11-07 05:27:00,2024-11-07 05:29:00
1,Arrivals,5:15,5:11,Liege,FedEx,Landed 05:07,07-11-24 23:55,Basel,05:07,2024-11-07 05:07:00,2024-11-07 05:11:00
2,Arrivals,5:15,5:14,Brussels,European Air Transport / DHL,Landed 05:11,07-11-24 23:55,Basel,05:11,2024-11-07 05:11:00,2024-11-07 05:14:00
3,Arrivals,5:20,5:18,Cologne Bonn,Sprintair,Landed 05:16,07-11-24 23:55,Basel,05:16,2024-11-07 05:16:00,2024-11-07 05:18:00
4,Arrivals,5:40,5:54,Cologne Bonn,Sprintair,Landed 05:51,07-11-24 23:55,Basel,05:51,2024-11-07 05:51:00,2024-11-07 05:54:00
...,...,...,...,...,...,...,...,...,...,...,...
2748,Departures,20:25,NaN,Barcelona,easyJet,Departed 20:21,09-11-24 23:55,Basel,20:21,2024-11-09 20:21:00,NaT
2749,Departures,20:40,NaN,Leipzig,European Air Transport / DHL,Departed 20:57,09-11-24 23:55,Basel,20:57,2024-11-09 20:57:00,NaT
2750,Departures,21:00,NaN,Lisbon,easyJet,Departed 21:00,09-11-24 23:55,Basel,21:00,2024-11-09 21:00:00,NaT
2751,Departures,21:10,NaN,London Gatwick,easyJet,Departed 21:13,09-11-24 23:55,Basel,21:13,2024-11-09 21:13:00,NaT


In [172]:
df["ActualDate"].isna().sum()

np.int64(0)

In [173]:
# Handle missing airlines
df['Airline'] = df['Airline'].replace(["", " ", "None", "N/A"], np.nan)
df['Airline'] = df['Airline'].fillna('Unknown')

# For 'Departures', fill 'Expected' with 'Actual' if 'Expected' is empty.
df.loc[(df['Type'] == 'Departures') & (df['ExpectedDate'].isna()) & df['ActualDate'].notna(), 'ExpectedDate'] = df['ActualDate']
df.loc[(df['Type'] == 'Departures') & df['ActualDate'].isna(), ['ExpectedDate', 'ActualDate']] = np.nan

In [174]:
df[df["ExpectedDate"].isna()]

,Type,Time,Expected,Origin/Destination,Airline,Status,Date Scraped,LocalAirport,Actual,ActualDate,ExpectedDate


In [175]:
# Data enrichment - Calculate the time delay in minutes and add 'Delay' as a new column
df['Delay'] = (df['ActualDate'] - df['ExpectedDate']).dt.total_seconds() / 60  # Delay in minutes
df

,Type,Time,Expected,Origin/Destination,Airline,Status,Date Scraped,LocalAirport,Actual,ActualDate,ExpectedDate,Delay
0,Arrivals,5:15,5:29,Paris Charles De Gaulle,FedEx,Landed 05:27,07-11-24 23:55,Basel,05:27,2024-11-07 05:27:00,2024-11-07 05:29:00,-2.0
1,Arrivals,5:15,5:11,Liege,FedEx,Landed 05:07,07-11-24 23:55,Basel,05:07,2024-11-07 05:07:00,2024-11-07 05:11:00,-4.0
2,Arrivals,5:15,5:14,Brussels,European Air Transport / DHL,Landed 05:11,07-11-24 23:55,Basel,05:11,2024-11-07 05:11:00,2024-11-07 05:14:00,-3.0
3,Arrivals,5:20,5:18,Cologne Bonn,Sprintair,Landed 05:16,07-11-24 23:55,Basel,05:16,2024-11-07 05:16:00,2024-11-07 05:18:00,-2.0
4,Arrivals,5:40,5:54,Cologne Bonn,Sprintair,Landed 05:51,07-11-24 23:55,Basel,05:51,2024-11-07 05:51:00,2024-11-07 05:54:00,-3.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2748,Departures,20:25,NaN,Barcelona,easyJet,Departed 20:21,09-11-24 23:55,Basel,20:21,2024-11-09 20:21:00,2024-11-09 20:21:00,0.0
2749,Departures,20:40,NaN,Leipzig,European Air Transport / DHL,Departed 20:57,09-11-24 23:55,Basel,20:57,2024-11-09 20:57:00,2024-11-09 20:57:00,0.0
2750,Departures,21:00,NaN,Lisbon,easyJet,Departed 21:00,09-11-24 23:55,Basel,21:00,2024-11-09 21:00:00,2024-11-09 21:00:00,0.0
2751,Departures,21:10,NaN,London Gatwick,easyJet,Departed 21:13,09-11-24 23:55,Basel,21:13,2024-11-09 21:13:00,2024-11-09 21:13:00,0.0


In [176]:
# Rename the column
df.rename(columns={'Origin/Destination': 'ForeignAirport','ExpectedDate':'PlannedDate'}, inplace=True)

# Ensure columns are in the desired order
df = df[['Type','LocalAirport','ForeignAirport','Airline','PlannedDate','ActualDate','Delay']]

# Save the updated DataFrame to a new file
df.to_csv('./data/combined_cleaned_flight_data.csv', index=False)

# Print the DataFrame to verify the changes
print(df)

            Type LocalAirport           ForeignAirport  \
0       Arrivals        Basel  Paris Charles De Gaulle   
1       Arrivals        Basel                    Liege   
2       Arrivals        Basel                 Brussels   
3       Arrivals        Basel             Cologne Bonn   
4       Arrivals        Basel             Cologne Bonn   
...          ...          ...                      ...   
2748  Departures        Basel                Barcelona   
2749  Departures        Basel                  Leipzig   
2750  Departures        Basel                   Lisbon   
2751  Departures        Basel           London Gatwick   
2752  Departures        Basel                     Doha   

                           Airline         PlannedDate          ActualDate  \
0                            FedEx 2024-11-07 05:29:00 2024-11-07 05:27:00   
1                            FedEx 2024-11-07 05:11:00 2024-11-07 05:07:00   
2     European Air Transport / DHL 2024-11-07 05:14:00 2024-11-07 05: